# 11 · Capstone: `drift-cam` Prototype

## What this notebook covers
An end-to-end production anomaly detection pipeline for image streams, built as the `drift-cam` prototype: CLIP/DINOv2 embeddings + streaming semantic drift detection + anomaly scoring with PatchCore-style nearest-neighbour scoring.

## System design
```
Image stream → Feature extraction (backbone) → Embedding store
                                             ↓
                                   k-NN anomaly scoring
                                             ↓
                               ADWIN drift monitor on scores
                                             ↓
                              Alert: "semantic drift detected"
```

The key insight: monitor the *distribution of anomaly scores* over a sliding window. A sudden increase in average anomaly score (or a shift in the score distribution) indicates that the incoming stream has drifted away from the training distribution — either because of legitimate data distribution change or because of a real increase in defects/anomalies.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from collections import deque
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

# ── Synthetic image stream ──────────────────────────────────────────────────────
def synthetic_image(size=64, anomaly=False, drift=False):
    img = np.random.randint(80, 140, (size, size, 3), dtype=np.uint8)
    img[::10, :] = 160; img[:, ::10] = 160  # grid texture
    if drift:
        img[:, :, 0] = np.clip(img[:, :, 0].astype(int) + 40, 0, 255).astype(np.uint8)
    if anomaly:
        cx, cy = np.random.randint(10, 55, 2); r = np.random.randint(4, 12)
        yy, xx = np.ogrid[:size, :size]
        mask = (xx-cx)**2 + (yy-cy)**2 <= r**2
        img[mask] = np.random.randint(0, 30)
    return Image.fromarray(img)

# Stream: 600 images (0-399 normal, 400-599 drifted with 30% anomaly rate)
stream_images, stream_labels, stream_is_drift = [], [], []
for i in range(400):
    is_anom = np.random.rand() < 0.03
    stream_images.append(synthetic_image(anomaly=is_anom))
    stream_labels.append(int(is_anom))
    stream_is_drift.append(0)
for i in range(200):
    is_anom = np.random.rand() < 0.30  # higher anomaly rate post-drift
    stream_images.append(synthetic_image(anomaly=is_anom, drift=True))
    stream_labels.append(int(is_anom))
    stream_is_drift.append(1)

stream_labels = np.array(stream_labels)
print(f"Stream: {len(stream_images)} images  |  anomalies: {stream_labels.sum()} ({stream_labels.mean():.1%})")
print(f"Pre-drift: {stream_labels[:400].mean():.1%} anomaly rate  |  Post-drift: {stream_labels[400:].mean():.1%}")

In [ ]:
# ── Feature extractor ─────────────────────────────────────────────────────────
transform = T.Compose([
    T.Resize((64, 64)), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class ImgDataset(Dataset):
    def __init__(self, imgs, tf): self.imgs = imgs; self.tf = tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tf(self.imgs[i])

weights = ResNet18_Weights.DEFAULT
backbone = resnet18(weights=weights)
feature_extractor = nn.Sequential(*list(backbone.children())[:-1])  # remove classifier
feature_extractor.eval()

def get_embeddings(images, batch_size=32):
    loader = DataLoader(ImgDataset(images, transform), batch_size=batch_size)
    embeds = []
    with torch.no_grad():
        for batch in loader:
            out = feature_extractor(batch).squeeze(-1).squeeze(-1)
            embeds.append(out.numpy())
    return np.concatenate(embeds, axis=0)

print("Extracting stream embeddings...")
all_embeddings = get_embeddings(stream_images)
print(f"Embeddings: {all_embeddings.shape}")

In [ ]:
# ── Build memory bank from first 200 normal images ─────────────────────────────
normal_mask = (stream_labels[:200] == 0)
memory_bank = all_embeddings[:200][normal_mask]
print(f"Memory bank: {memory_bank.shape[0]} normal embeddings")

nn_index = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn_index.fit(memory_bank)

# Score all stream images
dists, _ = nn_index.kneighbors(all_embeddings)
anomaly_scores = dists.mean(axis=1)

auc = roc_auc_score(stream_labels, anomaly_scores)
print(f"k-NN anomaly detection AUROC: {auc:.4f}")

In [ ]:
# ── ADWIN drift monitor on anomaly score stream ─────────────────────────────────
class ADWIN:
    def __init__(self, delta=0.002):
        self.delta = delta
        self.window = deque()
        self.drift_points = []

    def update(self, value, t):
        self.window.append(value)
        w = list(self.window)
        n = len(w)
        if n < 4: return False
        for split in range(1, n):
            w0, w1 = np.array(w[:split]), np.array(w[split:])
            eps = np.sqrt((1/(2*len(w0)) + 1/(2*len(w1))) * np.log(4*n/self.delta))
            if abs(w0.mean() - w1.mean()) >= eps:
                for _ in range(split):
                    if self.window: self.window.popleft()
                self.drift_points.append(t)
                return True
        return False

adwin = ADWIN(delta=0.001)
drift_events = []
for t, score in enumerate(anomaly_scores):
    if adwin.update(score, t):
        drift_events.append(t)

print(f"ADWIN drift events: {len(drift_events)}")
print(f"First drift detected at: {drift_events[:3] if drift_events else 'None'}")
print(f"True drift begins at: 400")

In [ ]:
# ── Dashboard ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Anomaly scores with true labels
window = 30
rolling_avg = np.convolve(anomaly_scores, np.ones(window)/window, mode="valid")
axes[0].plot(anomaly_scores, color="lightgrey", lw=0.5, alpha=0.8)
axes[0].plot(np.arange(window-1, len(anomaly_scores)), rolling_avg, color="steelblue", lw=1.5, label=f"Rolling mean (w={window})")
axes[0].scatter(np.where(stream_labels==1)[0], anomaly_scores[stream_labels==1], c="red", s=10, zorder=5, alpha=0.6, label="True anomaly")
for de in drift_events[:5]:
    axes[0].axvline(de, color="orange", alpha=0.7, lw=1.5)
axes[0].axvline(400, color="black", linestyle="--", lw=1.5, label="True drift onset")
axes[0].set_title("Anomaly scores over stream (orange lines = ADWIN alerts)")
axes[0].legend(fontsize=8)

# Score distribution: pre vs post drift
axes[1].hist(anomaly_scores[:400],    bins=40, alpha=0.6, color="steelblue", density=True, label="Pre-drift")
axes[1].hist(anomaly_scores[400:],    bins=40, alpha=0.6, color="tomato",    density=True, label="Post-drift")
axes[1].set_title("Score distribution: pre vs post drift")
axes[1].legend(); axes[1].set_xlabel("Anomaly score")

# ROC-AUC rolling window
window_auc = 100
rolling_aucs = []
for i in range(window_auc, len(stream_labels)):
    sl = stream_labels[i-window_auc:i]
    sc = anomaly_scores[i-window_auc:i]
    if sl.sum() > 0 and (sl == 0).sum() > 0:
        rolling_aucs.append(roc_auc_score(sl, sc))
    else:
        rolling_aucs.append(np.nan)
axes[2].plot(np.arange(window_auc, len(stream_labels)), rolling_aucs, color="seagreen", lw=1.5)
axes[2].axvline(400, color="black", linestyle="--", lw=1.5, label="True drift onset")
axes[2].set_title(f"Rolling AUROC (window={window_auc})")
axes[2].set_xlabel("Stream index"); axes[2].set_ylabel("AUROC")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{base}/11_driftcam_capstone.png", dpi=120)
plt.show()
print(f"\nFinal overall AUROC: {auc:.4f}")